In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import f1_score, roc_auc_score
from lightgbm import LGBMClassifier

#路徑設定
CLEAN_TRAIN = "data/clean_train.csv"
CLEAN_TEST  = "data/clean_test.csv"
SAMPLE_SUB  = "data/sample_submission.csv"

# 讀檔 
clean_train = pd.read_csv(CLEAN_TRAIN, low_memory=False)
clean_test  = pd.read_csv(CLEAN_TEST,  low_memory=False)
sample_sub  = pd.read_csv(SAMPLE_SUB,  low_memory=False)

print("clean_train:", clean_train.shape)
print("clean_test :", clean_test.shape)
print("sample_sub :", sample_sub.shape)
display(sample_sub.head())

clean_train: (97962, 32)
clean_test : (3084, 29)
sample_sub : (1001, 4)


,rally_uid,serverGetPoint,pointId,actionId
0,15432,NaN,NaN,NaN
1,15436,NaN,NaN,NaN
2,15440,NaN,NaN,NaN
3,15444,NaN,NaN,NaN
4,15447,NaN,NaN,NaN


In [2]:
# 基本欄位與對齊鍵值確認

KEY = "rally_uid"
assert KEY in clean_test.columns and KEY in sample_sub.columns, f"❌ 找不到對齊鍵 {KEY}"

# 評分三欄：Task1 actionId (多分類)、Task2 pointId (多分類)、Task3 serverGetPoint (二分類機率)
ACTION_COL = "next_actionId"   # 訓練標籤（由前處理建立）
POINT_COL  = "next_pointId"    # 訓練標籤（由前處理建立）
# 任務3的目標名稱可能不同（依清理檔而定），常見有 'serverGetPoint' 或 'server_won_point'
RALLY_TARGET_CANDIDATES = ["serverGetPoint", "server_won_point"]
RALLY_TARGET = next((c for c in RALLY_TARGET_CANDIDATES if c in clean_train.columns), None)

for must_col in [ACTION_COL, POINT_COL]:
    if must_col not in clean_train.columns:
        raise KeyError(f"❌ clean_train 缺少必要標籤欄位：{must_col}，請回到前處理確認 next_* 是否建立成功。")

print(f"✅ 偵測到任務標籤：{ACTION_COL}, {POINT_COL}, 任務3目標欄位：{RALLY_TARGET}")

# 群組欄位（避免洩漏：以比賽或回合作為 fold 切分單位）
GROUP_COL = "match_id" if "match_id" in clean_train.columns else ("rally_id" if "rally_id" in clean_train.columns else None)
if GROUP_COL is None:
    print("⚠️ 找不到 match_id / rally_id，將退化為隨機切分（較不理想）。")

✅ 偵測到任務標籤：next_actionId, next_pointId, 任務3目標欄位：serverGetPoint


In [3]:
# 特徵挑選
# 基礎特徵（
base_feats = [
    # 近期 K 擊（若前處理有建立 prev1~prev3）
    "prev1_actionId","prev2_actionId","prev3_actionId",
    "prev1_pointId","prev2_pointId","prev3_pointId",
    # 比分與回合進度
    "score_diff","rally_progress","rally_length",
    # 身分/性別（不同資料版本欄位名稱可能不同，列出常見候選再過濾）
    "sex","gamePlayerId","gamePlayerOtherId","player_self_id","player_opponent_id"
]
# 也把任何 delta_* 類型自動加入（若有）
delta_feats = [c for c in clean_train.columns if c.startswith("delta_")]
features_all = [c for c in base_feats + delta_feats if c in clean_train.columns]

if len(features_all) == 0:
    raise ValueError("❌ 找不到可用特徵（prevK_*/score_diff/rally_* 等），請檢查前處理。")

print(f"✅ 可用特徵數：{len(features_all)}")
print(sorted(features_all))

✅ 可用特徵數：12
['gamePlayerId', 'gamePlayerOtherId', 'prev1_actionId', 'prev1_pointId', 'prev2_actionId', 'prev2_pointId', 'prev3_actionId', 'prev3_pointId', 'rally_length', 'rally_progress', 'score_diff', 'sex']


In [4]:
# ==== Cell K1: 共用工具（純 KNN；含標準化與可選 PCA；支援 GroupKFold） ====
import numpy as np
import pandas as pd
from typing import List, Optional, Tuple, Dict

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import f1_score, roc_auc_score

def build_knn_pipeline(n_neighbors:int=31, use_pca:bool=True, pca_var:float=0.95) -> Pipeline:
    steps = [("scaler", StandardScaler())]
    if use_pca:
        steps.append(("pca", PCA(n_components=pca_var, svd_solver="full", random_state=42)))
    steps.append(("knn", KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights="distance",      # 距離加權，對不平衡較穩
        metric="minkowski", p=2, # 歐氏距離
        n_jobs=-1
    )))
    return Pipeline(steps)

def cv_knn_macro_f1(
    X: pd.DataFrame, y: pd.Series, groups: Optional[pd.Series]=None,
    k_list: List[int]=[11, 21, 31, 41, 61],
    use_pca: bool=True, pca_var: float=0.95,
    n_splits:int=5, random_state:int=42, verbose:bool=True
) -> Tuple[int, Dict[int, float]]:
    scores = {}
    if groups is not None:
        gkf = GroupKFold(n_splits=n_splits)
        split_iter = list(gkf.split(X, y, groups))
    else:
        X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=random_state, stratify=y)
        split_iter = [(X_tr.index, X_va.index)]

    for k in k_list:
        fold_scores = []
        for fold, (tr_idx, va_idx) in enumerate(split_iter, start=1):
            pipe = build_knn_pipeline(n_neighbors=k, use_pca=use_pca, pca_var=pca_var)
            pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            pred = pipe.predict(X.iloc[va_idx])
            f1 = f1_score(y.iloc[va_idx], pred, average="macro")
            fold_scores.append(f1)
            if verbose:
                print(f"[k={k:>3} | fold {fold}] Macro F1 = {f1:.4f}")
        scores[k] = float(np.mean(fold_scores))
        if verbose:
            print(f"[k={k:>3}] CV Macro F1 mean = {scores[k]:.4f}")
    best_k = max(scores, key=lambda t: scores[t])
    if verbose:
        print(f"Best k = {best_k} (Macro F1 = {scores[best_k]:.4f})")
    return best_k, scores

def cv_knn_auc_binary(
    X: pd.DataFrame, y: pd.Series, groups: Optional[pd.Series]=None,
    k_list: List[int]=[11, 21, 31, 41, 61],
    use_pca: bool=True, pca_var: float=0.95,
    n_splits:int=5, random_state:int=42, verbose:bool=True
) -> Tuple[int, Dict[int, float]]:
    scores = {}
    if groups is not None:
        gkf = GroupKFold(n_splits=n_splits)
        split_iter = list(gkf.split(X, y, groups))
    else:
        X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=random_state, stratify=y)
        split_iter = [(X_tr.index, X_va.index)]

    for k in k_list:
        fold_scores = []
        for fold, (tr_idx, va_idx) in enumerate(split_iter, start=1):
            pipe = build_knn_pipeline(n_neighbors=k, use_pca=use_pca, pca_var=pca_var)
            pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            proba = pipe.predict_proba(X.iloc[va_idx])[:, 1]
            auc = roc_auc_score(y.iloc[va_idx], proba)
            fold_scores.append(auc)
            if verbose:
                print(f"[k={k:>3} | fold {fold}] AUC = {auc:.4f}")
        scores[k] = float(np.mean(fold_scores))
        if verbose:
            print(f"[k={k:>3}] CV AUC mean = {scores[k]:.4f}")
    best_k = max(scores, key=lambda t: scores[t])
    if verbose:
        print(f"Best k = {best_k} (AUC = {scores[best_k]:.4f})")
    return best_k, scores


In [5]:
# ==== Cell K2: Task 1 (next_actionId) — 純 KNN ====
ACTION_COL = globals().get("ACTION_COL", "next_actionId")
GROUP_COL  = globals().get("GROUP_COL",  "match_id")

X_action_knn = clean_train[features_all].copy().fillna(0)
y_action_knn = clean_train[ACTION_COL].astype(int)
groups = clean_train[GROUP_COL] if (GROUP_COL in clean_train.columns) else None

best_k_act, scores_act = cv_knn_macro_f1(
    X_action_knn, y_action_knn, groups=groups,
    k_list=[11, 21, 31, 41, 61],
    use_pca=True, pca_var=0.95,   # 如要完全不做 PCA，改成 False
    n_splits=5, random_state=42, verbose=True
)

pipe_action_knn = build_knn_pipeline(n_neighbors=best_k_act, use_pca=True, pca_var=0.95)
pipe_action_knn.fit(X_action_knn, y_action_knn)
print(f"[Task1-KNN] best_k={best_k_act}, 訓練完成。")


[k= 11 | fold 1] Macro F1 = 0.1476
[k= 11 | fold 2] Macro F1 = 0.1507
[k= 11 | fold 3] Macro F1 = 0.1326
[k= 11 | fold 4] Macro F1 = 0.1499
[k= 11 | fold 5] Macro F1 = 0.1518
[k= 11] CV Macro F1 mean = 0.1465
[k= 21 | fold 1] Macro F1 = 0.1409
[k= 21 | fold 2] Macro F1 = 0.1454
[k= 21 | fold 3] Macro F1 = 0.1278
[k= 21 | fold 4] Macro F1 = 0.1424
[k= 21 | fold 5] Macro F1 = 0.1430
[k= 21] CV Macro F1 mean = 0.1399
[k= 31 | fold 1] Macro F1 = 0.1383
[k= 31 | fold 2] Macro F1 = 0.1355
[k= 31 | fold 3] Macro F1 = 0.1230
[k= 31 | fold 4] Macro F1 = 0.1323
[k= 31 | fold 5] Macro F1 = 0.1340
[k= 31] CV Macro F1 mean = 0.1326
[k= 41 | fold 1] Macro F1 = 0.1339
[k= 41 | fold 2] Macro F1 = 0.1322
[k= 41 | fold 3] Macro F1 = 0.1203
[k= 41 | fold 4] Macro F1 = 0.1312
[k= 41 | fold 5] Macro F1 = 0.1323
[k= 41] CV Macro F1 mean = 0.1300
[k= 61 | fold 1] Macro F1 = 0.1282
[k= 61 | fold 2] Macro F1 = 0.1258
[k= 61 | fold 3] Macro F1 = 0.1174
[k= 61 | fold 4] Macro F1 = 0.1270
[k= 61 | fold 5] Macro F

In [6]:
# ==== Cell K3: Task 2 (next_pointId) — 純 KNN ====
POINT_COL = globals().get("POINT_COL", "next_pointId")

X_point_knn = clean_train[features_all].copy().fillna(0)
y_point_knn = clean_train[POINT_COL].astype(int)
groups = clean_train[GROUP_COL] if (GROUP_COL in clean_train.columns) else None

best_k_pt, scores_pt = cv_knn_macro_f1(
    X_point_knn, y_point_knn, groups=groups,
    k_list=[11, 21, 31, 41, 61],
    use_pca=True, pca_var=0.95,
    n_splits=5, random_state=42, verbose=True
)

pipe_point_knn = build_knn_pipeline(n_neighbors=best_k_pt, use_pca=True, pca_var=0.95)
pipe_point_knn.fit(X_point_knn, y_point_knn)
print(f"[Task2-KNN] best_k={best_k_pt}, 訓練完成。")


[k= 11 | fold 1] Macro F1 = 0.1494
[k= 11 | fold 2] Macro F1 = 0.1482
[k= 11 | fold 3] Macro F1 = 0.1389
[k= 11 | fold 4] Macro F1 = 0.1469
[k= 11 | fold 5] Macro F1 = 0.1496
[k= 11] CV Macro F1 mean = 0.1466
[k= 21 | fold 1] Macro F1 = 0.1488
[k= 21 | fold 2] Macro F1 = 0.1449
[k= 21 | fold 3] Macro F1 = 0.1362
[k= 21 | fold 4] Macro F1 = 0.1467
[k= 21 | fold 5] Macro F1 = 0.1437
[k= 21] CV Macro F1 mean = 0.1441
[k= 31 | fold 1] Macro F1 = 0.1452
[k= 31 | fold 2] Macro F1 = 0.1406
[k= 31 | fold 3] Macro F1 = 0.1274
[k= 31 | fold 4] Macro F1 = 0.1440
[k= 31 | fold 5] Macro F1 = 0.1408
[k= 31] CV Macro F1 mean = 0.1396
[k= 41 | fold 1] Macro F1 = 0.1434
[k= 41 | fold 2] Macro F1 = 0.1391
[k= 41 | fold 3] Macro F1 = 0.1226
[k= 41 | fold 4] Macro F1 = 0.1434
[k= 41 | fold 5] Macro F1 = 0.1385
[k= 41] CV Macro F1 mean = 0.1374
[k= 61 | fold 1] Macro F1 = 0.1380
[k= 61 | fold 2] Macro F1 = 0.1327
[k= 61 | fold 3] Macro F1 = 0.1207
[k= 61 | fold 4] Macro F1 = 0.1322
[k= 61 | fold 5] Macro F

In [7]:
# ==== Cell K4: Task 3 (serverGetPoint) — 純 KNN ====
# 自動找二分類目標欄位
CANDIDATES = [
    globals().get("RALLY_TARGET", None),
    "serverGetPoint", "server_won_point", "server_get_point", "rally_outcome"
]
RALLY_TARGET = next((c for c in CANDIDATES if isinstance(c, str) and c in clean_train.columns), None)

if RALLY_TARGET is None:
    print("⚠️ 找不到二分類目標欄位，跳過 Task3-KNN。")
    pipe_rally_knn, best_k_rally, features_rally_knn = None, None, None
else:
    # 避免爭議：不使用 prevK_*（如要實驗可改回保留）
    remove_prefixes = ("prev1_", "prev2_", "prev3_", "prev4_", "prev5_")
    features_rally_knn = [f for f in features_all if not f.startswith(remove_prefixes)]
    if len(features_rally_knn) == 0:
        fallback = [c for c in ["score_diff","rally_progress","rally_length"] if c in clean_train.columns]
        features_rally_knn = fallback if len(fallback) else [c for c in clean_train.columns if c != RALLY_TARGET]

    X_rally_knn = clean_train[features_rally_knn].copy().fillna(0)
    y_raw = clean_train[RALLY_TARGET]
    uniq = pd.Series(y_raw.dropna().unique())
    if len(uniq) != 2:
        raise ValueError(f"{RALLY_TARGET} 非二元標籤（unique={list(uniq)}）")

    # 將標籤映成 0/1（保持穩定）
    uniq_sorted = sorted(uniq.tolist(), key=lambda v: str(v))
    y_rally_knn = y_raw.map({uniq_sorted[0]:0, uniq_sorted[1]:1}).astype(int)

    groups = clean_train[GROUP_COL] if (GROUP_COL in clean_train.columns) else None

    best_k_rally, scores_rally = cv_knn_auc_binary(
        X_rally_knn, y_rally_knn, groups=groups,
        k_list=[11, 21, 31, 41, 61],
        use_pca=True, pca_var=0.95,
        n_splits=5, random_state=42, verbose=True
    )

    pipe_rally_knn = build_knn_pipeline(n_neighbors=best_k_rally, use_pca=True, pca_var=0.95)
    pipe_rally_knn.fit(X_rally_knn, y_rally_knn)
    print(f"[Task3-KNN] best_k={best_k_rally}, 訓練完成。")


[k= 11 | fold 1] AUC = 0.5019
[k= 11 | fold 2] AUC = 0.5025
[k= 11 | fold 3] AUC = 0.5024
[k= 11 | fold 4] AUC = 0.5118
[k= 11 | fold 5] AUC = 0.4922
[k= 11] CV AUC mean = 0.5021
[k= 21 | fold 1] AUC = 0.5039
[k= 21 | fold 2] AUC = 0.5010
[k= 21 | fold 3] AUC = 0.5024
[k= 21 | fold 4] AUC = 0.5094
[k= 21 | fold 5] AUC = 0.4981
[k= 21] CV AUC mean = 0.5030
[k= 31 | fold 1] AUC = 0.5051
[k= 31 | fold 2] AUC = 0.5020
[k= 31 | fold 3] AUC = 0.5032
[k= 31 | fold 4] AUC = 0.5120
[k= 31 | fold 5] AUC = 0.4983
[k= 31] CV AUC mean = 0.5041
[k= 41 | fold 1] AUC = 0.5034
[k= 41 | fold 2] AUC = 0.5033
[k= 41 | fold 3] AUC = 0.5062
[k= 41 | fold 4] AUC = 0.5135
[k= 41 | fold 5] AUC = 0.5013
[k= 41] CV AUC mean = 0.5055
[k= 61 | fold 1] AUC = 0.5011
[k= 61 | fold 2] AUC = 0.5031
[k= 61 | fold 3] AUC = 0.5059
[k= 61 | fold 4] AUC = 0.5135
[k= 61 | fold 5] AUC = 0.5027
[k= 61] CV AUC mean = 0.5053
Best k = 41 (AUC = 0.5055)
[Task3-KNN] best_k=41, 訓練完成。


In [10]:
# ==== Cell K5: 生成 submission（純 KNN 推論） ====
# Task1 / Task2 features
X_test_all = clean_test[features_all].copy().fillna(0)

# --- Task 1: next_actionId ---
pred_action_knn = pipe_action_knn.predict(X_test_all)

# --- Task 2: next_pointId ---
pred_point_knn  = pipe_point_knn.predict(X_test_all)

# --- Task 3: serverGetPoint ---
# 若前面沒有訓練到 Task3（找不到欄位），就跳過
if 'pipe_rally_knn' in globals() and pipe_rally_knn is not None:
    # 和訓練時的 features_rally_knn 一致
    X_test_rally = clean_test[features_rally_knn].copy().fillna(0)

    proba_rally_knn = pipe_rally_knn.predict_proba(X_test_rally)[:, 1]  # 機率版
    pred_rally_knn  = (proba_rally_knn >= 0.5).astype(int)              # 0/1 版
else:
    # 若沒有 Task3，建立空欄避免報錯（視賽規而定）
    proba_rally_knn = None
    pred_rally_knn = None

# --- 組裝提交 ---
sub_cols = ["rallyuid", "next_actionId", "next_pointId", "serverGetPoint"]
sub = pd.DataFrame({
    "rallyuid": clean_test["rallyuid"],
    "next_actionId": pred_action_knn,
    "next_pointId":  pred_point_knn,
    # 這裡預設上傳 0/1；若賽規要求「機率」，請改成 proba_rally_knn
    "serverGetPoint": pred_rally_knn if pred_rally_knn is not None else 0
})
sub.to_csv("submission_knn_label.csv", index=False)

# 若賽規要機率（AUC 通常更適合機率）
if proba_rally_knn is not None:
    sub_prob = sub.copy()
    sub_prob["serverGetPoint"] = proba_rally_knn
    sub_prob.to_csv("submission_knn_prob.csv", index=False)

print("✅ 已輸出：submission_knn_label.csv（0/1）",
      "與 submission_knn_prob.csv（機率，若有 Task3）")


KeyError: 'rallyuid'